In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
import boto3
import datetime as dt

try:
    import snowflake.connector
except:
    ! pip install snowflake-connector-python
    import snowflake.connector

from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives.asymmetric import rsa, dsa
from cryptography.hazmat.primitives import serialization

try:
    import optbinning
except:
    ! pip install optbinning

try:
    import catboost
except:
    ! pip install catboost

In [ ]:
dtm_now = dt.datetime.now()
print(f'Latest run date: {dtm_now}')

#### Functions

In [ ]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # init client
    cls_client = boto3.client(
        's3',
    )
    # download file
    cls_client.download_file(
        str_project, 
        str_bucket_path, 
        str_local_path,
    )

In [ ]:
def get_tier(flt_ecnl, dict_tiers):
    if flt_ecnl <= dict_tiers['A1']:
        return 'A1'
    elif flt_ecnl <= dict_tiers['A']:
        return 'A'
    elif flt_ecnl <= dict_tiers['B']:
        return 'B'
    elif flt_ecnl <= dict_tiers['C']:
        return 'C'
    elif flt_ecnl <= dict_tiers['D']:
        return 'D'
    else:
        return 'Decline'

#### Constants

In [ ]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# output
str_dirname_output = './output'

# inst
list_str_inst = [
    # from ben: 2025-02-21
    'CURRENT',
    'SELF',
    # key words
    'CHIME-STRIDE',
    'CHIMEFINAL',
    # from dustin: 2025-02-24
    'SELF FIN',
    'SELF/LEAD',
    'SELFINC/LEAD',
    'SBNASELFLNDR',
    'SBNA SELF',
    'CHIME',
    'CLEO',
    'CLEO AI',
    'VARO',
    'ATLAS',
    'ATLCAPBKSELF',
    'POSSIBLE',
    'POSSIBLE FIN',
    'KIKOFF',
    'SUPER.COM',
    'STEP',
    'STEP MOBILE',
    'BRIGHT',
    'BRIGHT BLDR',
    'FIG TECH INC',
    'SELF/RENT',
    'SELFBILLSE',
    'PROGRESSRES',
    'FLEX',
    'FLEXFINANCE',
]

#### Make output dir

In [ ]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Connect to snowflake

In [ ]:
# load 
str_filename = 'datascience_rsa_key.p8'
str_local_path = f'./{str_filename}'
with open(str_local_path, "rb") as key:
    p_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend(),
    )
# convert to bytes
private_key = p_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)
# connect to snowflake
conn = snowflake.connector.connect(
    user='datascience', 
    private_key=private_key,
    account='pfs', 
    warehouse='datascience',
    database='raw',
    schema='source_s3_scorehistory',
)

#### Load Gen 13 data

In [ ]:
str_filename = 'df.gzip'
str_uri = f's3://20241112-simple-model-test/08_prep_data/{str_filename}'
df = pd.read_parquet(str_uri)
df['applicationdate__app'] = pd.to_datetime(df['applicationdate__app'])
df['dealerstampcreation__app'] = pd.to_datetime(df['dealerstampcreation__app'])
# show
df

#### Load Gen 13 model monitoring data

In [ ]:
str_filename = 'df.gzip'
str_uri = f's3://20250121-gen-13-model-monitoring/04_concatenate_files/{str_filename}'
df_tmp = pd.read_parquet(str_uri)
df_tmp['applicationdate__app'] = pd.to_datetime(df_tmp['applicationdate__app'])
df_tmp['dealerstampcreation__app'] = pd.to_datetime(df_tmp['dealerstampcreation__app'])
# show
df_tmp

#### Find tags

In [ ]:
list_cols = [col for col in df_tmp.columns if 'tag' in col and col != 'subjectage__ln']
list_cols.append('sum')
# drop
df_tmp.drop(list_cols, axis=1, inplace=True)
# show
df_tmp

#### Concatenate data

In [ ]:
%%time

# concat
df = pd.concat([df, df_tmp])
# get date
df['request_datetime'] = df['request_datetime'].apply(
    lambda x: str(x)[:10],
)
# make datetime
df['request_datetime'] = pd.to_datetime(df['request_datetime'])
# set dtype
df['accountid'] = df['accountid'].astype(int)
# sort
df.sort_values(by='request_datetime', ascending=True, inplace=True)
# rm dup rows
df.drop_duplicates(subset=['accountid','bitdebtor'], keep='last', inplace=True)
# show
df

#### Get funded apps

In [ ]:
# query funded
str_filename = 'query_funded.sql'
str_local_path = f'./query/{str_filename}'
str_query = open(str_local_path, 'r').read()

# pull payloads
df_tmp = pd.read_sql(
    sql=str_query,
    con=conn,
)
# set as int
df_tmp['ACCOUNT_NUMBER'] = df_tmp['ACCOUNT_NUMBER'].astype(int)
# list
list_int_account = list(df_tmp['ACCOUNT_NUMBER'])
# save memory
del df_tmp
list_int_account = list(dict.fromkeys(list_int_account))
# len
int_n_funded = len(list_int_account)
print(f'There are {int_n_funded} funded apps')

#### Subset to funded

In [ ]:
%%time

df['funded'] = df['accountid'].apply(
    lambda x: 1 if x in list_int_account else 0,
)
# subset
df = df[df['funded'] == 1].copy()
# show
df

#### Get DLv2 apps

In [ ]:
# query funded
str_filename = 'query_direct.sql'
str_local_path = f'./query/{str_filename}'
str_query = open(str_local_path, 'r').read()

# pull payloads
df_tmp = pd.read_sql(
    sql=str_query,
    con=conn,
)
# set as int
df_tmp['ACCOUNTID'] = df_tmp['ACCOUNTID'].astype(int)
# list
list_int_account = list(df_tmp['ACCOUNTID'])
# save memory
del df_tmp
list_int_account = list(dict.fromkeys(list_int_account))
# len
int_n_direct = len(list_int_account)
print(f'There are {int_n_direct} direct apps')

#### Subset to indirect

In [ ]:
%%time

df['direct'] = df['accountid'].apply(
    lambda x: 1 if x in list_int_account else 0,
)
# subset
df = df[df['direct'] == 0].copy()
# show
df

#### Create Chime tags

In [ ]:
df['list_institutions'] = df['str_institution__tu_pmthx'].apply(
    lambda x: eval(x.replace('nan','None')),
)
df['list_institutions'] = df['list_institutions'].apply(
    lambda x: [] if x is None else x,
)
list_str_col_new = []
for str_inst in tqdm(list_str_inst):
    str_col_new = f'{str_inst}_tag'
    df[str_col_new] = df['list_institutions'].apply(
        lambda x: 1 if str_inst in x else 0,
    )
    list_str_col_new.append(str_col_new)
# sum of tags
df['sum'] = df[list_str_col_new].sum(axis=1)
# has a credit builder tag
df['has_inst_tag'] = df['sum'].apply(
    lambda x: 1 if x > 0 else 0,
)
flt_mn = df['has_inst_tag'].mean()
print(f'Proportion has tag: {flt_mn:0.4f}')

#### Drop chime tags that don't show up

In [ ]:
list_cols = []
for col in tqdm(list_str_col_new):
    # get sum
    int_sum = df[col].sum()
    # logic
    if int_sum == 0:
        list_cols.append(col)
    else:
        pass
int_len = len(list_cols)
print(f'There were {int_len} chime institutions not in the data')
# drop
df.drop(list_cols, axis=1, inplace=True)
# show
df

#### Engineer

In [ ]:
# payment hx
df['ENG-wtd_avg'] = df['flt_wtd_avg_open__tu_pmthx'].fillna(df['flt_wtd_avg_closed__tu_pmthx'])

In [ ]:
# ltv
df['ENG-loan_to_value'] = df['amtfinanced__app'] / df['bookvalue__app']

In [ ]:
# bk
df['ENG-bk'] = df['intopenbktype__app'].apply(
    lambda x: 1 if pd.notnull(x) else 0,
)

#### Convert non-numeric to string

In [ ]:
for col in tqdm(df.columns):
    str_dtype = df[col].dtype
    if str_dtype not in ['int64','float64']:
        df[col] = df[col].astype(str)
    else:
        pass

#### Write to s3

In [ ]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(
    str_uri,
    compression='gzip',
)